# BioEmu2 C-GAIMMD on chignolin

This tutorial runs C-GAIMMD with BioEmu2 as the generator on the folding of
chignolin (CLN025, YYDPETGTWY) in explicit water, with GROMACS. BioEmu2 is a
pretrained generative model of protein structure ensembles. C-GAIMMD keeps it
frozen and conditions it on the committor with small adapters on its trunk
embeddings; a committor head reads BioEmu2's own features. Every round the
fit trains a fresh head on the shots, then fresh adapters on frames labelled by
that head.

The steps:

1. the system: the MD files, the generator atoms (the 166 protein atoms) and
   the BioEmu2 system folder (`build-system`: the atom map between the MD and
   BioEmu2, the completion of the hydrogens BioEmu2 does not model, and the
   trunk embeddings);
2. the params file with the sequential fit and the candidate preparation:
   the polish, re-solvation and minimization;
3. the initial network: the committor head trained on unbiased frames of the
   folded and unfolded states;
4. the preflight;
5. a local launch and a cluster job script;
6. a restart after a stop, and monitoring.

> **Outputs pending.** This tutorial needs a GPU and GROMACS, the `bioemu2-cuda` extra and the BioEmu2 weights in the Hugging Face cache. The page
> shows the code and the explanations; the outputs of a run on a GPU
> workstation will be added. Every path that depends on your machine is set in
> the first code cell.

## 0. Paths and settings

The system folder holds the solvated MD system and its unbiased MD: `run.gro`
and `topol.top` (CHARMM22* with TIP3P, with the force-field folder),
`run.mdp`, `em.mdp` and `relax.mdp` (production, minimization and a short NVT
relaxation, all without a `-DPOSRES` define: the preparation tools add their
own restraints), `states.py` (the fraction of native contacts Q: folded A,
unfolded B), and unbiased MD of each state (`free_A.xtc`, `free_B.xtc`). Set
the paths for your machine.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import aimmd
import matplotlib.pyplot as plt
import MDAnalysis as mda
import numpy as np

# --- configure ---------------------------------------------------------------------
SYSTEM = Path("/path/to/chignolin_system")         # run.gro topol.top *.itp run.mdp em.mdp relax.mdp states.py
FREE_MD = {"A": SYSTEM / "free_A.xtc", "B": SYSTEM / "free_B.xtc"}   # unbiased MD of each state
CAMPAIGN = Path("chignolin_cgaimmd").resolve()     # params, system folder, pretrained network
RUN = Path("/local/scratch/chignolin_cgaimmd_run") # the run folder, on a local disk
GMX = shutil.which("gmx") or "/path/to/gromacs/bin/gmx"
THREADS = 12                                       # OpenMP threads per GROMACS run
os.environ.setdefault("HF_HUB_OFFLINE", "1")       # the weights come from the local cache
# ------------------------------------------------------------------------------------

CAMPAIGN.mkdir(parents=True, exist_ok=True)
for item in SYSTEM.iterdir():
    if item.suffix in (".gro", ".top", ".itp", ".mdp", ".py") or item.name.endswith(".ff"):
        target = CAMPAIGN / item.name
        if item.is_dir():
            shutil.copytree(item, target, dirs_exist_ok=True)
        else:
            shutil.copy(item, target)
os.chdir(CAMPAIGN)
sys.path.insert(0, str(CAMPAIGN))
from states import states_function                 # noqa: E402

## 1. The system

### The generator atoms and the files of the polish

The generator atoms are the protein: the first `N_PROTEIN` atoms of
`run.gro`. The validity check and its polish (a restrained minimization of
each raw sample) work on the protein alone, so they need a protein-only
structure and topology, and clash references of unbiased protein-only frames.

In [ ]:
u = mda.Universe("run.gro")
N_PROTEIN = u.select_atoms("protein").n_atoms
assert (u.select_atoms("protein").indices == np.arange(N_PROTEIN)).all(), "the protein must come first"
u.atoms[:N_PROTEIN].write("protein.gro")


def solute_topology(source, target, solvent=("SOL", "WAT", "HOH", "NA", "CL", "SOD", "CLA", "K")):
    """A copy of `source` whose [ molecules ] keeps only the molecules that are not solvent or ions."""
    lines, section = [], None
    for line in Path(source).read_text().splitlines():
        stripped = line.split(";")[0].strip()
        if stripped.startswith("["):
            section = stripped.strip("[] ").lower()
        elif section == "molecules" and stripped and stripped.split()[0] in solvent:
            continue
        lines.append(line)
    Path(target).write_text("\n".join(lines) + "\n")


solute_topology("topol.top", "topol_protein.top")

# clash references: every 50th frame of the unbiased MD, protein only
with mda.Writer("clash_reference_protein.xtc", N_PROTEIN) as writer:
    for state in "AB":
        md_u = mda.Universe("run.gro", str(FREE_MD[state]))
        for ts in md_u.trajectory[::50]:
            writer.write(md_u.atoms[:N_PROTEIN])
print(N_PROTEIN, "protein atoms")

### The BioEmu2 system folder

`python -m aimmd.cgaimmd.bioemu2 build-system` writes everything about the
system that a BioEmu2 run needs besides the params file:

- the **atom map** between the MD atoms and BioEmu2's atoms (protein chains
  residue by residue and by atom name, translating force-field names such as
  the C-terminal `OT1`);
- the **completion** of the generator atoms BioEmu2 does not model (here the
  hydrogens and the second C-terminal oxygen), placed from the MD structure by
  rigid fits of nearby atoms;
- the stereocentres, the reference pose and the token pools of the feature
  readout;
- the **trunk embeddings** of BioEmu2 for this sequence, computed once, with
  the MSA (query-only here) and the checkpoint recorded.

The BioEmu2 system is a small YAML file; its sequence is read from the MD
structure.

In [ ]:
ONE_LETTER = {"ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "ASH": "D", "CYS": "C", "CYX": "C",
              "GLN": "Q", "GLU": "E", "GLH": "E", "GLY": "G", "HIS": "H", "HSD": "H", "HSE": "H",
              "HSP": "H", "HID": "H", "HIE": "H", "HIP": "H", "ILE": "I", "LEU": "L", "LYS": "K",
              "LYN": "K", "MET": "M", "PHE": "F", "PRO": "P", "SER": "S", "THR": "T", "TRP": "W",
              "TYR": "Y", "VAL": "V"}
sequence = "".join(ONE_LETTER[res.resname] for res in u.select_atoms("protein").residues)
Path("bioemu2_system.yaml").write_text(
    "entities:\n- entity_type: protein\n  asym_ids: [A]\n  sequence: " + sequence + "\n")
print("sequence:", sequence)

result = subprocess.run(
    [sys.executable, "-m", "aimmd.cgaimmd.bioemu2", "build-system",
     "--bioemu2-system", "bioemu2_system.yaml", "--md-structure", "run.gro",
     "--topology", "topol.top", "--n-atoms", str(N_PROTEIN), "--out", "system",
     "--msa", "query-only", "--device", "cuda"],
    capture_output=True, text=True)
print(result.stdout[-2000:], result.stderr[-2000:])

## 2. The params file

Written in two steps again: without `initial_network` first, to build the
network the pretraining trains. The parts that differ from a PaiNN run:

- the network is a `BioEmu2Network` built from the system folder; its default
  generator is `'bioemu2_edm'` (200 EDM steps, no guidance), so
  `cgaimmd_generator` is left unset;
- the validity check is `PolishedValidity`: each raw sample is checked,
  polished by a restrained minimization of the protein alone, and checked
  again; raw BioEmu2 samples often have near-contacts that the bond-graph check
  reads as extra bonds;
- `GromacsSolvator` puts each candidate back into the solvated system
  (solvation, ions, restrained minimization, 20 ps of restrained NVT), and
  `GromacsMinimizer` minimizes the prepared frame with restraints;
- the fit is `sequential_fit`: a fresh committor head every round, and fresh
  adapters once 100 shooting points exist;
- free simulations start at a random in-state frame of the seed files and
  later from the state's accumulated frames, so that the unfolded state is
  sampled, not only its edge.

In [ ]:
PARAMS = f'''
"""BioEmu2 C-GAIMMD on solvated chignolin (tutorial 5)."""
import numpy as np

from aimmd.cgaimmd.bioemu2 import BioEmu2Network, sequential_fit
from aimmd.cgaimmd.preparation.descriptors import GeneratorAtoms, GeneratorDescriptors
from aimmd.cgaimmd.preparation.minimize import GromacsMinimizer
from aimmd.cgaimmd.preparation.solvate import GromacsSolvator, NVTStage
from aimmd.cgaimmd.preparation.topology import read_masses
from aimmd.cgaimmd.preparation.validity import PolishedValidity, TopologyValidity
from states import states_function

GMX = {GMX!r}
THREADS = {THREADS}
N_PROTEIN = {N_PROTEIN}
engine = 'gromacs'
chain_type = 'cgaimmd'
states = 'ARB'                                   # A folded, B unfolded
topology = 'run.gro'
gmx_mdp = 'run.mdp'
gmx_grompp = f'{{GMX}} grompp -maxwarn 5 -p topol.top'
gmx_mdrun = f'{{GMX}} mdrun -ntmpi 1 -ntomp {{THREADS}}'
gmx_eneconv = ''
initial_paths = ['initial_A.xtc', 'initial_B.xtc']
free_seeding_position = 'random'
free_restart_source = 'equilibrium'
round_shooting_index = True
masses_from_topology = True
retry_with_state_definition_glitches = True
gen_temperature = 340.0
max_length = 20000
reweight_parameters = {{'free_threshold': 20}}
cgaimmd_parameters = {{'n_targets': 16, 'candidates_per_target': 2, 'generation_seed': 1}}

DESCRIPTORS = GeneratorDescriptors('run.gro', GeneratorAtoms(n_solute=N_PROTEIN),
                                   masses=read_masses('topol.top'))


def descriptors_function(trajectory):
    return DESCRIPTORS(trajectory)


class Network(BioEmu2Network):
    def __init__(self):
        super().__init__('system')


network = Network()

_protein = GeneratorDescriptors('protein.gro', GeneratorAtoms(n_solute=N_PROTEIN))
_topology_validity = TopologyValidity('topol_protein.top', _protein,
                                      node_labels=np.arange(N_PROTEIN),   # BioEmu2 keeps topology order
                                      stereocenters=network.system.stereocenters(),
                                      clash_reference=['clash_reference_protein.xtc'])
_polisher = GromacsMinimizer(em_mdp='em.mdp', restrain='generator_heavy',
                             topology='topol_protein.top', grompp=f'{{GMX}} grompp -maxwarn 5',
                             mdrun=f'{{GMX}} mdrun -ntmpi 1 -ntomp 1 -nb cpu -pin off')
_validity = PolishedValidity(_topology_validity, _polisher, 'protein.gro', workers=THREADS)


def cgaimmd_validity_function(positions, context):
    return _validity(positions, context)


_solvator = GromacsSolvator(DESCRIPTORS, topology='topol.top', em_mdp='em.mdp',
                            nvt_stages=[NVTStage('relax.mdp', 20.0, gen_vel=True)],
                            grompp=f'{{GMX}} grompp', mdrun=f'{{GMX}} mdrun -ntmpi 1 -ntomp {{THREADS}}',
                            maxwarn=5)


def cgaimmd_prepare_function(positions, context):
    return _solvator(positions, context)


_minimizer = GromacsMinimizer(em_mdp='em.mdp', restrain='generator_heavy', restraint_k=1e4,
                              emtol=100.0, nsteps=1000, grompp=f'{{GMX}} grompp',
                              mdrun=f'{{GMX}} mdrun -ntmpi 1 -ntomp {{THREADS}}', maxwarn=5)


def cgaimmd_minimize_function(frame, context):
    return _minimizer(frame, context)


def fit(params, pathensemble, verbose=False, worker=None):
    return sequential_fit(params, pathensemble, verbose=True, worker=worker,
                          min_shots_for_generator=100, adapter_steps=3000)
'''

# the per-state seed files: every 200th frame of the unbiased MD
for state in "AB":
    md_u = mda.Universe("run.gro", str(FREE_MD[state]))
    with mda.Writer(f"initial_{state}.xtc", md_u.atoms.n_atoms) as writer:
        for ts in md_u.trajectory[::200]:
            writer.write(md_u.atoms)

Path("params.py").write_text(PARAMS.lstrip())
params = aimmd.Params.load("params.py", save=False)
print(params.network.system.summary())

## 3. The initial network

`pretrain_initial_network` keeps the adapters at zero (the generator is plain
BioEmu2), fits the feature scaler of the committor head on the in-state
frames, and trains the head to separate the two states. Every later round
keeps this scaler and starts its head and adapters afresh.

In [ ]:
from aimmd.cgaimmd.bioemu2 import pretrain_initial_network
from aimmd.cgaimmd.flow_matching.pretrain import collect_end_state_frames

frames = collect_end_state_frames([str(FREE_MD["A"]), str(FREE_MD["B"])], structure="run.gro",
                                  descriptors_function=params.descriptors_function,
                                  states_function=states_function, states="ARB", stride=20,
                                  max_frames=20000)
codes = np.where(frames["state"] == "A", 1, 2)          # 1: first state, 2: last state
record = pretrain_initial_network(params.network.to("cuda"), frames["coordinates"], codes,
                                  network_file="pretrained/networkARB.h5", verbose=True)
print({key: record[key] for key in record if key not in ("settings",)})

with open("params.py", "a") as handle:
    handle.write("\ninitial_network = 'pretrained/networkARB.h5'\n")
params = aimmd.Params.load("params.py", save=False)

## 4. The preflight

On a GPU: the generator check, one BioEmu2 generation, the polish, the
re-solvation and minimization of a candidate, the final check, and GROMACS.
The fit check validates the keywords of `sequential_fit` without training.

In [ ]:
result = subprocess.run([sys.executable, "-m", "aimmd.cgaimmd.preflight", "params.py",
                         "--candidates", "4", "--device", "cuda"], capture_output=True, text=True)
print(result.stdout)
print("exit status:", result.returncode)

## 5. Launch

### Locally

One GPU is enough for a test: two shooting chains, the free simulations and
the trainer share it. The candidate preparation runs GROMACS on the CPU cores.

In [ ]:
launcher = aimmd.Launcher(params, str(RUN))
launcher.run(2, 1, 1, nsteps=40, walltime=4 * 3600)

### As a cluster job

On a GH200 cluster such as JUPITER (four GPUs per node), two nodes carry five
shooting chains, the two free simulations and the trainer, one GPU each. The
preamble loads the modules, activates the environment and keeps Hugging Face
offline. Write the script on the cluster itself, in the job's environment.

In [ ]:
params.slurm_header = "#SBATCH --account=<project>\n#SBATCH --partition=<gpu partition>"
launcher = aimmd.Launcher(params, str(RUN))
launcher.create_job(
    "job.sh", 5, 1, 1, walltime=12 * 3600, ntasks_per_node=4, cpus_per_task=THREADS,
    gpus_per_task=1, skip_binding=False,
    preamble=["module load Stages/2026 nvidia-compilers/25.9-CUDA-13 OpenMPI/5.0.8 GCC/14.3.0",
              "module load GROMACS/2025.4-CUDA-13",
              "source /path/to/project/cgaimmd_venv/bin/activate",
              "export HF_HOME=/path/to/project/cache/huggingface HF_HUB_OFFLINE=1"],
    preflight=True)
print(Path("job.sh").read_text())

## 6. Restarts and monitoring

Jobs end at their time limit; resubmit the same script to continue
(`sbatch job.sh`), or call `launcher.run(...)` again locally. The trainer
continues its round count from `networkARB.json`; a trainer stopped during a
round keeps that round's network as `networkARB.final.h5`. An interrupted shot
resumes from its checkpoint with its in-flight record.

The sidecars record the BioEmu2 checkpoint, its Hugging Face snapshot and the
parameter checksum of the frozen model, so every shot can be traced to the
model that generated it.

In [ ]:
from aimmd.cgaimmd.sidecar import read_shot_sidecar, shot_table, summarise_shots

rows = shot_table(RUN, params=params)
summary = summarise_shots(rows, frame_interval_ps=5.0)
print("shots:", summary["n_shots"], "| reactive share:", summary["reactive_share"],
      "| MD in shots:", summary["md_ns_in_shots"], "ns")
print("rejected in the refills:", summary["stage1_rejected"])
print("rejected in stage 2:", summary["candidates_rejected"])

last = read_shot_sidecar(str(RUN / rows[-1]["chain"] / rows[-1]["path"]))
print(json.dumps(last["network"]["provenance"], indent=1))
print(json.dumps(last["generator"], indent=1))

## Where to go from here

- Tutorial 6 adds a ligand: trypsin with benzamidine, with the ligand mapped by
  its chemical graph and a system-specific validity check.
- {doc}`../cgaimmd_production` covers installs on clusters, backups and
  post-training; {doc}`../cgaimmd_troubleshooting` lists the messages of the
  checks.